# CGC + Chiller Test

This test is for testing the water supply from the chiller to all water cooled cgc devices.

## Import, Setup Logger, Create Instances

In [1]:
# Import required modules
import sys
import os
import logging
from datetime import datetime
from pathlib import Path

# Add src to path
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'src'))

from devices.cgc.psu.psu import PSU
from devices.cgc.sw.sw import SW
from devices.cgc.sw_HR.sw_HR import SWHR
from devices.chiller.chiller import Chiller, ChillerCommands

In [2]:
# Setup external logger
repo_root = Path(os.getcwd()).parent.parent
log_dir = repo_root / "debugging" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = log_dir / f"016_Temp_Check_GC_Chiller_{timestamp}.log"

logger = logging.getLogger(f"016_Temp_Check_GC_Chiller_{timestamp}")
logger.setLevel(logging.DEBUG)

file_handler = logging.FileHandler(log_file)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
logger.addHandler(console_handler)

logger.info("Logger initialized.")
print(f"Log file: {log_file}")

2026-03-23 12:46:15,085 - INFO - Logger initialized.


Log file: C:\Users\ESIBDlab\PycharmProjects\esibd_bs\debugging\logs\016_Temp_Check_GC_Chiller_20260323_124615.log


## Create Device Instances

In [3]:
psu1 = PSU("psu1", com=3, port=0, logger=logger)

In [4]:
psu2 = PSU("psu2", com=4, port=1, logger=logger)

In [5]:
psu3 = PSU("psu3", com=5, port=2, logger=logger)

In [6]:
psu4 = PSU("psu4", com=6, port=3, logger=logger)

In [7]:
swA = SWHR("swA", com=7, stream=0, logger=logger)

In [8]:
swB = SW("swB", com=8, port=0, logger=logger)

In [9]:
chiller = Chiller("cgc_chiller", port="COM23", baudrate=115200, logger=logger)

2026-03-23 12:46:23,040 - INFO - Using external logger for device 'cgc_chiller'


## Connect All Devices

In [10]:
psu1.connect()
psu1.set_comspeed(230400)

2026-03-23 12:46:25,650 - INFO - psu1 - Connecting to PSU device psu1 on COM3, port 0
2026-03-23 12:46:25,799 - INFO - psu1 - Successfully connected to PSU device psu1


0

In [11]:
psu2.connect()
psu2.set_comspeed(230400)

2026-03-23 12:46:26,603 - INFO - psu2 - Connecting to PSU device psu2 on COM4, port 1
2026-03-23 12:46:26,750 - INFO - psu2 - Successfully connected to PSU device psu2


0

In [12]:
psu3.connect()
psu3.set_comspeed(230400)

2026-03-23 12:46:32,627 - INFO - psu3 - Connecting to PSU device psu3 on COM5, port 2
2026-03-23 12:46:36,010 - INFO - psu3 - Successfully connected to PSU device psu3


0

In [13]:
psu4.connect()
psu4.set_comspeed(230400)

2026-03-23 12:46:38,037 - INFO - psu4 - Connecting to PSU device psu4 on COM6, port 3
2026-03-23 12:46:38,184 - INFO - psu4 - Successfully connected to PSU device psu4


0

In [14]:
swA.connect()
swA.set_comspeed(230400)

2026-03-23 12:46:39,461 - INFO - swA - Connecting to SW_HR device swA on COM7, stream 0
2026-03-23 12:46:39,608 - INFO - swA - Successfully connected to SW_HR device swA (baud rate: 230400)


(0, 230400)

In [15]:
swB.connect()
swB.set_comspeed(230400)

2026-03-23 12:46:40,430 - INFO - swB - Connecting to SW device swB on COM8, port 0
2026-03-23 12:46:40,578 - INFO - swB - Successfully connected to SW device swB (baud rate: 230400)


(0, 230400)

In [16]:
chiller.connect()

2026-03-23 12:46:41,300 - INFO - Connecting to chiller cgc_chiller on COM23


True

## Configure and Start Chiller

In [ ]:
chiller.set_temperature(8.0)

In [ ]:
chiller.start_device()

## Temperature Monitoring Loop

In [17]:
psu1.get_sensor_data()

(0, 18.61, 18.47, 18.53)

In [18]:
psu2.get_sensor_data()

(0, 18.62, 18.19, 18.02)

In [19]:
psu3.get_sensor_data()

(0, 18.1, 19.09, 18.52)

In [20]:
psu4.get_sensor_data()

(0, 18.99, 18.63, 18.5)

In [21]:
swA.get_sensor_data()

(0, 18.57, 18.66, 0.21)

In [22]:
swB.get_sensor_data()

(0, 0.21, 18.46, 18.4)

In [23]:
chiller.read_temp()

17.45

In [ ]:
import time

interval = 1  # seconds
try:
    while True:
        for psu in [psu1, psu2, psu3, psu4]:
            status, t0, t1, t2 = psu.get_sensor_data()
            if status == psu.NO_ERR:
                logger.info(f"{psu.device_id} Sensor0={t0:.1f} Sensor1={t1:.1f} Sensor2={t2:.1f} degC")

        for sw in [swA, swB]:
            status, t0, t1, t2 = sw.get_sensor_data()
            if status == sw.NO_ERR:
                logger.info(f"{sw.device_id} Sensor0={t0:.1f} Sensor1={t1:.1f} Sensor2={t2:.1f} degC")

        temp = chiller.read_temp()
        if temp is not None:
            logger.info(f"{chiller.device_id} Temp={temp:.2f} degC")

        time.sleep(interval)
except KeyboardInterrupt:
    logger.info("Temperature monitoring stopped")

## Stop Chiller and Disconnect

In [ ]:
chiller.stop_device()

In [ ]:
psu1.disconnect()
psu2.disconnect()
psu3.disconnect()
psu4.disconnect()
swA.disconnect()
swB.disconnect()
chiller.disconnect()